# Protein Graphs and GNN-LRP\n\nThis notebook shows the end-to-end biological flow: download a real PDB structure, build a residue graph, load the protein classifier, and compute GNN-LRP scores.

In [ ]:
import sys\nfrom pathlib import Path\nimport torch\n\nPROJECT_ROOT = Path.cwd()\nif not (PROJECT_ROOT / 'README.md').exists():\n    PROJECT_ROOT = PROJECT_ROOT.parent\nsys.path.insert(0, str(PROJECT_ROOT))\nsys.path.insert(0, str(PROJECT_ROOT / '07_gnn_lrp'))\n\nfrom utils.pdb_graphs import download_pdb_file, build_residue_graph_from_pdb\nfrom utils.dense_gcn import DenseGCNGraphClassifier\nfrom gnn_lrp import explain_graph_with_gnn_lrp

In [ ]:
pdb_path = download_pdb_file('1LYZ', PROJECT_ROOT / 'data' / 'pdb')\ngraph = build_residue_graph_from_pdb(pdb_path, chain_id='A')\ngraph.num_nodes(), graph.residue_labels()[:5]

In [ ]:
checkpoint = PROJECT_ROOT / 'artifacts' / 'protein_classifier' / 'protein_classifier.pt'\nmodel = DenseGCNGraphClassifier(\n    in_features=graph.node_features.shape[1],\n    hidden_features=32,\n    num_classes=2,\n    num_layers=2,\n)\nmodel.load_state_dict(torch.load(checkpoint, map_location='cpu'))\nexplanation = explain_graph_with_gnn_lrp(model, graph.adjacency, graph.node_features)\nexplanation.class_probability

In [ ]:
top_indices = explanation.node_relevance.argsort()[::-1][:10]\n[(graph.residue_labels()[index], float(explanation.node_relevance[index])) for index in top_indices]

For a complete workflow with saved plots and PyMOL export, use `08_case_study/run_case_study.py` from the command line.